# Paper 19 · Batch Normalization

**Citation:** Sergey Ioffe, Christian Szegedy, “Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift” (2015).

**Paper:** https://arxiv.org/abs/1502.03167

> **Scale gap:** We compare deep MLP training with and without BatchNorm on 8×8 digits.

## Mathematical Framework

Before reproducing the paper experimentally, work through:

- [Math 04 · Statistics & Likelihood](../../math/04_statistics_likelihood.ipynb)
- [Math 06 · Optimization](../../math/06_optimization.ipynb)
- [Math 10 · Neural-Network Mathematics](../../math/10_neural_network_math.ipynb)

Your explanation should connect the paper's empirical claim to its **mathematical objective, representation, assumptions, and optimization/statistical argument**.

## Before you read
1. What differs between BatchNorm training and evaluation behavior?
2. Why do running statistics exist?
3. Which parts of the original explanatory story remain debated versus the empirical utility?

## Central claim
Normalizing intermediate activations using mini-batch statistics can substantially ease optimization and permit more aggressive training.

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
d=load_digits(); X=torch.tensor((d.data/16).astype("float32")); y=torch.tensor(d.target.astype("int64"))
tr,te=train_test_split(np.arange(len(y)),test_size=.3,random_state=0,stratify=y.numpy())
Xtr,Xte,ytr,yte=X[tr],X[te],y[tr],y[te]

## Deep MLP variants

In [ ]:
class Net(nn.Module):
    def __init__(self,bn):
        super().__init__(); layers=[]; D=64
        for _ in range(6):
            layers.append(nn.Linear(D,D))
            if bn: layers.append(nn.BatchNorm1d(D))
            layers.append(nn.ReLU())
        self.body=nn.Sequential(*layers); self.out=nn.Linear(D,10)
    def forward(self,x): return self.out(self.body(x))
def train(bn):
    torch.manual_seed(0); m=Net(bn); opt=torch.optim.SGD(m.parameters(),lr=.15); ce=nn.CrossEntropyLoss(); hist=[]
    for _ in range(60):
        m.train(); opt.zero_grad(); loss=ce(m(Xtr),ytr); loss.backward(); opt.step()
        m.eval()
        with torch.no_grad(): acc=(m(Xte).argmax(1)==yte).float().mean().item()
        hist.append([loss.item(),acc])
    return np.array(hist)
a=train(False); b=train(True)
print("no BN acc",a[-1,1],"BN acc",b[-1,1])
plt.plot(a[:,0],label="no BN"); plt.plot(b[:,0],label="BatchNorm"); plt.yscale("log"); plt.legend(); plt.show()

### Ablation
Increase learning rate until the non-BN model becomes unstable. Does BatchNorm extend the stable range?

## Ablation table

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
1. What problem existed before this work?
2. What was actually new?
3. What evidence did this notebook reproduce?
4. What does the scale gap prevent you from claiming?
5. Which contribution remains important today?
6. What would you test next?